# 03 - UHI metrics (Phase 3)

**Colombo UHI practicum.** Turns the Phase 2 temperature layer into urban heat
island quantities:

1. **SUHII** (mean urban LST - mean rural LST) for every year, every LST source,
   under **both** rural definitions, as one tidy table;
2. **UTFVI** with the six-class scheme, as three epoch maps and a per-year
   class-share series;
3. **per-scene z-scores** and a hot-pixel mask at 1 sigma and 2 sigma;
4. **per-GN and per-DS zonal statistics** - the table Phase 5 consumes;
5. **LST against NDVI / NDBI / MNDWI / built-up fraction**, per year, with
   coefficients, R2 and p-values.

Run top-to-bottom in **Google Colab** after `02_lst_pipeline.ipynb` has worked
once. All logic lives in `src/colombo_uhi/`; this notebook orchestrates and
displays.

> **Caveat (CLAUDE.md #1):** every number here is **LAND SURFACE TEMPERATURE**,
> never air temperature. A "hot pixel" below is a hot roof or road surface, not
> a temperature a resident feels. Surface UHI can be roughly 2x the canopy-air UHI.
>
> **Caveat (CLAUDE.md #2):** never read a SUHII, a class share or a division mean
> without the pixel count beside it. Every table here carries one.
>
> **Caveat (CLAUDE.md #4):** Landsat sees one ~10:30 local overpass. Night-time
> UHI comes only from MODIS - and MODIS night LST is accepted at up to 3 K stated
> uncertainty against 1 K for day, so **night SUHII is weaker evidence than day
> SUHII** and must never be presented as an equal-confidence pair.
>
> **Caveat (CLAUDE.md #5):** the two rural definitions differ **by construction**
> (CMC vs a 15-25 km annulus, against built vs vegetated LCZ classes inside the
> district). Their SUHII values will differ. That spread **is** the result; it is
> not a discrepancy to reconcile away.

### Two things about UTFVI that are routinely misread

**It is a ratio, so it depends on the temperature scale.** The class breaks
(0.005, 0.010, ...) assume **degrees Celsius**: with Tmean around 30 degC the
0.005 break is about 0.15 degC. On Kelvin the same break would mean about 1.5 K.
`uhi_metrics.utfvi()` refuses anything but the Celsius band for this reason.

**Its reference moves with the data.** Tmean is *that year's own* spatial mean
(`uhi.utfvi.reference = per_year_aoi_mean`, the standard formulation). So UTFVI
describes **where** heat sits within a year, not how hot the year was. A city
that warms uniformly by 2 degC produces identical UTFVI classes before and after.
**Never read epoch-to-epoch class drift as warming** - that is what the Phase 4
Mann-Kendall and Sen's slope products measure.

### If Earth Engine says "User memory limit exceeded"

Same rules as notebook 02 - it is graph depth, not pixel count. The Phase 3
specifics, in order of effect:

1. **Pass `water=STATIC_WATER` when building the rural masks** (Step 1). The
   default `aoi.water_exclusion_mask` composites Landsat internally, and that
   composite is re-instantiated for *every* image it masks. Across 26 years x 6
   sources that is the Colab run 3 failure, multiplied. Step 1 already does this.
2. **Shrink `WORK_REGION`.** It defaults to the union of both methods'
   `suhii_region()`s, which is already far tighter than `aoi.analysis_region`
   (Western Province + 25 km).
3. **Lower `uhi.suhii.batch_years`** (default 4, valid down to 1). Changes no
   numbers, only the number of round trips.
4. **Build the mask pairs once** and pass them to every source, as Step 2 does.
   Rebuilding them per source multiplies the LCZ mosaic and SRTM threshold.
5. **Drop sources.** `SUHII_SOURCES` below is a plain list; shorten it to get a
   first result, then run the full set.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00, 01 or 02 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "single_overpass", "sensitivity_reporting"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 1 - geometries, the cheap water mask, and both rural references

Three decisions are made in this cell and they set the cost of everything after it.

| Decision | Why |
|---|---|
| `WORK_REGION` = union of both methods' `suhii_region()` | The masks are clipped, so any enclosing region gives identical numbers. `aoi.analysis_region` (Western Province + 25 km) would cost roughly 20x the pixels for nothing. |
| `STATIC_WATER` instead of the combined water mask | The combined mask composites Landsat internally and re-instantiates that composite per masked image. Notebook 02 measured the substitution at **-0.074 degC** on the CMC 2025 mean - quote that figure, do not re-derive it. |
| Mask pairs built **once**, reused for all six sources | Each pair carries an LCZ mosaic, an SRTM threshold and a water mask. |

The printed mask areas are a **cross-check against Phase 1's Colab-verified
values**, not decoration. If they have moved, the water-mask substitution changed
the masks and that must be investigated before any SUHII number is trusted.

In [ ]:
# COLAB: RUN THIS CELL
import ee

from colombo_uhi import aoi, composites, indices, landsat, modis, uhi_metrics, viz

# Fail fast and legibly if a STALE colombo_uhi is loaded (see the purge above).
_required = {
    "aoi": ["rural_reference", "static_water_mask", "lcz_scope_geometry"],
    "composites": ["annual_composites", "warn_if_counts_are_empty"],
    "uhi_metrics": [
        "suhii_all_sources", "mask_pairs", "suhii_work_region", "utfvi",
        "utfvi_class_image", "utfvi_class_series", "lst_zscore",
        "hot_pixel_mask", "zonal_by_division", "driver_series",
    ],
    "viz": ["plot_suhii_sensitivity", "plot_utfvi_class_shares", "plot_lst_vs_index"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  uhi_metrics loaded from: {uhi_metrics.__file__}\n"
        "Fix, in order:\n"
        "  1. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  2. If it persists, your local commits are not pushed - check that the\n"
        "     HEAD line printed by the clone cell is the revision you expect."
    )

METHODS = uhi_metrics.resolve_methods(None, params)   # both rural definitions
print("Rural definitions:", METHODS)

district_fc = aoi.colombo_district(params)
district_geom = district_fc.geometry(10)

# Degrade gracefully if the CMC cannot be built, so the rest still runs
# (same pattern as notebooks 01 and 02).
try:
    cmc_geom = aoi.cmc_boundary(params)
    print("CMC boundary built.")
except Exception as exc:  # noqa: BLE001 - surface the real reason, keep going
    cmc_geom = None
    print("COULD NOT BUILD THE CMC:", exc)
    print("Falling back to Colombo District where the CMC is needed.")

zone_geom = cmc_geom if cmc_geom is not None else district_geom
zone_label = "CMC" if cmc_geom is not None else "Colombo District"

# THE memory knob for this notebook. Shrink it first if EE runs out of memory.
WORK_REGION = uhi_metrics.suhii_work_region(params, METHODS)
print("Zonal reference zone:", zone_label)
print("WORK_REGION area (km2):", round(aoi.area_km2(WORK_REGION).getInfo(), 1))

In [ ]:
# COLAB: RUN THIS CELL
# The cheap water mask, then both rural references built ONCE from it.
STATIC_WATER = aoi.static_water_mask(params, region=WORK_REGION)
PAIRS = uhi_metrics.mask_pairs(params, methods=METHODS, water=STATIC_WATER)

# Cross-check against Phase 1's Colab-verified mask areas (run 5, 2026-08-08).
# These are the acceptance criterion for the water-mask substitution: if a number
# has moved by more than a few percent, STOP and find out why before trusting any
# SUHII below.
_expected_km2 = {
    ("buffer_ring", "urban"): 37.7,
    ("buffer_ring", "rural"): 206.1,
    ("lcz_based", "urban"): 458.5,
    ("lcz_based", "rural"): 152.2,
}

_areas = ee.Dictionary({
    f"{_m}_{_r}": aoi.mask_area_km2(_mask, params, scale_m=100)
    for _m, _masks in PAIRS.items()
    for _r, _mask in zip(uhi_metrics.MASK_ROLES, _masks)
}).getInfo()

print(f"{'mask':<26}{'measured':>10}{'Phase 1':>10}{'diff %':>9}")
for _method in METHODS:
    for _role in uhi_metrics.MASK_ROLES:
        _got = _areas[f"{_method}_{_role}"]
        _want = _expected_km2.get((_method, _role))
        if _want:
            _delta = (_got - _want) / _want * 100.0
            _flag = "" if abs(_delta) < 5 else "   <-- CHECK THIS"
            print(f"{_method + '/' + _role:<26}{_got:>10.1f}{_want:>10.1f}{_delta:>8.1f}%{_flag}")
        else:
            print(f"{_method + '/' + _role:<26}{_got:>10.1f}{'-':>10}{'-':>9}")
print()
print("Areas measured at 100 m. CMC land area is SCALE-DEPENDENT (40.18 km2 at")
print("30 m, 37.70 km2 at 300 m), so always quote the scale with any area.")

## Step 2 - the SUHII table

Six LST sources x two rural definitions x 26 years, batched
`uhi.suhii.batch_years` years per request: roughly **40 round trips**. Expect
this cell to take several minutes and to print as it goes.

| Source | Overpass | Note |
|---|---|---|
| `landsat_dry` | ~10:30, Jan-Mar | 30 m, reduced at 100 m, **median** |
| `terra_day` | ~10:30 | 1 km, **mean**, strict QC (0/0) |
| `terra_night` | ~22:30 | the cleanest signal in the dataset - full CMC pixel coverage |
| `aqua_day` | ~13:30 | near peak heating; **no data before 2002-07** (Aqua launch) |
| `aqua_night` | ~01:30 | |
| `terra_day_relaxed` | ~10:30 | **required by Phase 2 sign-off**: strict day QC keeps only 3.7% of observations and fails hardest over the dense coastal core (CMC 13-23 of 40 pixels; District 92-95%) |

Two things you should expect to see, which are **not** bugs:

* **Aqua rows for 2000-2002 come back empty** with `urban_pixels = 0`. Aqua
  launched in July 2002. The years stay in the table so the series is complete
  and the gap is visible rather than silently dropped.
* **A `COMPLETELY EMPTY` / `mostly empty` warning** may fire for those Aqua years.
  That warning is the caveat-2 enforcement doing its job.

In [ ]:
# COLAB: RUN THIS CELL
# The full SUHII table. Shorten SUHII_SOURCES to get a first result faster.
import time

SUHII_SOURCES = [_entry["key"] for _entry in params["uhi"]["suhii"]["sources"]]
print("Sources:", SUHII_SOURCES)
print("Batch size (years/request):", params["uhi"]["suhii"]["batch_years"])
print()

_t0 = time.time()
suhii = uhi_metrics.suhii_all_sources(
    params,
    pairs=PAIRS,
    sources=SUHII_SOURCES,
    methods=METHODS,
    region=WORK_REGION,
    progress=True,
)
print(f"\nDone in {time.time() - _t0:.0f} s; {len(suhii)} rows.")

In [ ]:
# COLAB: RUN THIS CELL
# Save the tidy table, then summarise it per source and rural definition.
import pandas as pd

os.makedirs("data/outputs", exist_ok=True)
_csv = "data/outputs/suhii_2000_2025.csv"
suhii.to_csv(_csv, index=False)
print("Wrote", _csv)
print()

# Mean SUHII per source x rural definition, with the pixel counts it rests on.
_summary = (
    suhii.dropna(subset=["suhii"])
    .groupby(["source", "rural_definition"])
    .agg(
        years=("year", "count"),
        suhii_mean=("suhii", "mean"),
        suhii_min=("suhii", "min"),
        suhii_max=("suhii", "max"),
        urban_px=("urban_pixels", "median"),
        rural_px=("rural_pixels", "median"),
    )
    .round(2)
)
print(_summary.to_string())
print()
print("The SPREAD between the two rural definitions for one source is the")
print("sensitivity CLAUDE.md requires reported. It is not an error bar and the")
print("two are not two estimates of one number.")

## Step 3 - SUHII sensitivity figure, and a plausibility check

One colour per source, one line style per rural definition: **the vertical gap
between a solid and a dashed line of the same colour is the sensitivity**, read
directly off the page.

The check below is a real acceptance test, not a formality. Phase 2 already
observed that **CMC Terra-night runs about +2 degC above Colombo District
Terra-night in every year** - a nocturnal UHI signal visible before Phase 3
existed. So the buffer-ring night SUHII should land near **+1 to +2 degC**. A
negative night SUHII, or one above about +6 degC, means a mask is wrong rather
than that Colombo is remarkable.

In [ ]:
# COLAB: RUN THIS CELL
from IPython.display import Image, display

os.makedirs("figures", exist_ok=True)
_fig = viz.plot_suhii_sensitivity(
    suhii, "figures/suhii_sensitivity_2000_2025.png", params
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

In [ ]:
# COLAB: RUN THIS CELL
# Plausibility check against the Phase 2 nocturnal signal.
_night = suhii[
    (suhii["source"] == "terra_night")
    & (suhii["rural_definition"] == "buffer_ring")
].dropna(subset=["suhii"])

if _night.empty:
    print("NO terra_night / buffer_ring rows survived. Investigate before continuing:")
    print("  - check urban_pixels/rural_pixels in the table above")
    print("  - run modis.qc_class_histogram to see which QC class is being kept")
else:
    _mean = float(_night["suhii"].mean())
    print(f"Terra night, buffer_ring: mean SUHII {_mean:+.2f} degC "
          f"over {len(_night)} years "
          f"(range {_night['suhii'].min():+.2f} to {_night['suhii'].max():+.2f})")
    print()
    if _mean < 0:
        print("FAIL: a NEGATIVE night SUHII means the urban and rural masks are")
        print("      probably swapped, or the rural mask has fallen inside the city.")
    elif _mean > 6:
        print("FAIL: implausibly large. Check that the rural mask is not empty and")
        print("      that the elevation cap has not left only high inland pixels.")
    else:
        print("PASS: consistent with the ~+2 degC CMC-vs-District nocturnal")
        print("      difference measured in Phase 2.")
print()
print("REMINDER: night LST is accepted at up to 3 K stated uncertainty against")
print("1 K for day. Night SUHII is WEAKER evidence than day SUHII - report the")
print("asymmetry, never the two as an equal-confidence pair.")

## Step 4 - UTFVI class maps for three epochs

Three Landsat dry-season composites, one per epoch from `uhi.utfvi.epochs`
(2000s / 2010s / 2020s), each classified into the six-class scheme.

`UTFVI_REGION` sets what "the AOI" means, and it changes what the maps say:

* **Colombo District** (default) - Tmean is the district mean, so the city stands
  out against its rural surroundings. The CMC will be largely `Worst`; that is the
  expected and informative result.
* **CMC** - Tmean is the urban mean, so the map shows *intra-urban* structure and
  the classes rebalance around the city's own average.

Both are legitimate; they answer different questions. Change the one variable.

In [ ]:
# COLAB: RUN THIS CELL
# Epoch composites -> UTFVI -> six-class images. One Landsat collection, reused.
UTFVI_REGION = district_geom          # switch to zone_geom for the intra-urban view
UTFVI_REGION_LABEL = "Colombo District"
UTFVI_SOURCE = "landsat_dry"
UTFVI_SCALE_M = 100                   # Tmean reduction scale; 30 m is unnecessary here

_landsat_scenes = uhi_metrics.source_collection(
    UTFVI_SOURCE, params, region=WORK_REGION
)

EPOCHS = list(params["uhi"]["utfvi"]["epochs"])
print("Epochs:", EPOCHS, "| AOI:", UTFVI_REGION_LABEL)

epoch_classes = {}
for _epoch in EPOCHS:
    _start, _end = uhi_metrics.epoch_years(params, _epoch)
    _composite = uhi_metrics.epoch_composite(
        UTFVI_SOURCE, params, _epoch, collection=_landsat_scenes
    )
    _index = uhi_metrics.utfvi(
        _composite, params, UTFVI_REGION, scale_m=UTFVI_SCALE_M
    )
    epoch_classes[_epoch] = uhi_metrics.utfvi_class_image(_index, params)
    print(f"  {_epoch}: {_start}-{_end} built")

In [ ]:
# COLAB: RUN THIS CELL
# Render the three epoch maps.
_vis = viz.utfvi_vis_params(params)
_labels = uhi_metrics.utfvi_class_labels(params)
_outline = viz.outline_image(district_fc, "222222", width=1)

for _epoch, _classes in epoch_classes.items():
    _start, _end = uhi_metrics.epoch_years(params, _epoch)
    _path = viz.save_thumbnail(
        [_classes.clip(UTFVI_REGION).visualize(**_vis), _outline],
        UTFVI_REGION,
        f"figures/utfvi_classes_{_start}_{_end}.png",
    )
    print(f"{_epoch} ({_start}-{_end}) ->", _path)
    display(Image(filename=str(_path)))

print("Class colours, coolest to hottest:")
for _index, (_label, _colour) in enumerate(zip(_labels, params["uhi"]["utfvi"]["palette"])):
    print(f"  {_index}  #{_colour}  {_label}")

In [ ]:
# COLAB: RUN THIS CELL
# The interpretation guard. Read this before drawing any conclusion from the maps.
print("HOW TO READ THE THREE EPOCH MAPS")
print("=" * 70)
print("UTFVI is referenced to EACH EPOCH'S OWN mean LST (uhi.utfvi.reference =")
print("per_year_aoi_mean). So these maps show WHERE heat sat within each epoch,")
print("NOT how much hotter the city became.")
print()
print("A city that warmed uniformly by 2 degC would produce THREE IDENTICAL MAPS.")
print("Class drift between epochs is a REDISTRIBUTION of heat - the hot core")
print("spreading, or greening cooling one district relative to the others - and")
print("is never, on its own, evidence of warming.")
print()
print("Warming is measured in Phase 4, by Mann-Kendall and Sen's slope on the")
print("annual composite series, with Benjamini-Hochberg FDR correction.")
print()
print("These are also LAND SURFACE temperatures at a single ~10:30 overpass in")
print("the Jan-Mar dry window - not air temperature, not a daily mean.")

## Step 5 - UTFVI class shares, per year

The same six classes, tabulated year by year. Each year is classified against its
own mean, so this series shows how **concentrated** each year's heat was.

Read it against the `pixel_count` column: a dry-season year with heavy cloud rests
on far fewer classified pixels, and its shares are correspondingly noisier.

In [ ]:
# COLAB: RUN THIS CELL
_t0 = time.time()
utfvi_shares = uhi_metrics.utfvi_class_series(
    UTFVI_SOURCE,
    params,
    UTFVI_REGION,
    collection=_landsat_scenes,
    scale_m=UTFVI_SCALE_M,
    progress=True,
)
print(f"\nDone in {time.time() - _t0:.0f} s.")

_csv = "data/outputs/utfvi_class_shares_2000_2025.csv"
utfvi_shares.to_csv(_csv, index=False)
print("Wrote", _csv)
print()
print(utfvi_shares.round(1).to_string(index=False))

In [ ]:
# COLAB: RUN THIS CELL
_fig = viz.plot_utfvi_class_shares(
    utfvi_shares, "figures/utfvi_class_shares_2000_2025.png", params
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

## Step 6 - settle the z-score degrees of freedom

**This cell resolves a genuine open question, and its answer belongs in
`config/params.yaml` and `PROGRESS.md`.**

The Earth Engine documentation does not state whether `ee.Reducer.stdDev()` is
the population standard deviation (numpy `ddof=0`) or the sample one (`ddof=1`),
and a separate `ee.Reducer.sampleStdDev()` also exists - which *suggests*, but
does not prove, that `stdDev` is the population form. Rather than guess, measure
it on an array whose two answers are known and far apart.

The difference is of order 1/n and is immaterial over a scene of thousands of
pixels. It matters because `uhi_metrics.zscore_array` (unit-tested locally) and
`uhi_metrics.lst_zscore` (server-side) must be the *same* statistic, or the tests
are pinning something the pipeline does not do.

In [ ]:
# COLAB: RUN THIS CELL
import numpy as np

_probe = [10.0, 12.0, 14.0, 16.0]
_pop = float(np.std(_probe, ddof=0))
_samp = float(np.std(_probe, ddof=1))

_ee_values = ee.Dictionary({
    "stdDev": ee.List(_probe).reduce(ee.Reducer.stdDev()),
    "sampleStdDev": ee.List(_probe).reduce(ee.Reducer.sampleStdDev()),
}).getInfo()

print(f"probe array: {_probe}")
print(f"  numpy ddof=0 (population): {_pop:.10f}")
print(f"  numpy ddof=1 (sample):     {_samp:.10f}")
print(f"  ee.Reducer.stdDev():       {_ee_values['stdDev']:.10f}")
print(f"  ee.Reducer.sampleStdDev(): {_ee_values['sampleStdDev']:.10f}")
print()

_measured = 0 if abs(_ee_values["stdDev"] - _pop) < 1e-9 else 1
_configured = params["uhi"]["zscore"]["ddof"]
print(f"MEASURED: ee.Reducer.stdDev() matches ddof={_measured}")
print(f"CONFIGURED in params (uhi.zscore.ddof): {_configured}")
if _measured == _configured:
    print("MATCH - nothing to change. Record the measurement in PROGRESS.md so")
    print("nobody has to repeat it.")
else:
    print()
    print(f"MISMATCH. Edit config/params.yaml: set uhi.zscore.ddof to {_measured},")
    print("commit it, re-run from the clone cell, and note the result in")
    print("PROGRESS.md. Until then the local unit tests pin a different")
    print("statistic from the one this pipeline computes.")

## Step 7 - per-scene z-scores and hot pixels

A z-score says how unusual a pixel is **within its own scene**, which makes dates
comparable in a way raw LST is not - and, for the same reason, cannot show that
one whole scene was hotter than another.

A "hot pixel" here is hot relative to the rest of this scene. It is **not** a
heat-health threshold, and it is still land surface temperature: a hot roof or
road, not an air temperature anyone experiences.

In [ ]:
# COLAB: RUN THIS CELL
Z_YEAR = 2025          # display year; not an analysis constant
Z_SCALE_M = 100

_composite = composites.annual_composites(
    _landsat_scenes,
    params,
    reducer="median",
    months=params["time"]["seasons"]["dry_window"]["months"],
    with_percentile=False,
    start_year=Z_YEAR,
    end_year=Z_YEAR,
).first()

_z = uhi_metrics.lst_zscore(
    _composite, params, UTFVI_REGION, scale_m=Z_SCALE_M
)

_areas = ee.Dictionary({
    f"sigma_{_s}": aoi.mask_area_km2(
        uhi_metrics.hot_pixel_mask(
            _composite, params, UTFVI_REGION, sigma=_s, scale_m=Z_SCALE_M
        ).selfMask(),
        params,
        scale_m=Z_SCALE_M,
    )
    for _s in params["uhi"]["zscore"]["sigma_options"]
}).getInfo()

print(f"Hot-pixel area over {UTFVI_REGION_LABEL}, dry season {Z_YEAR}, at {Z_SCALE_M} m:")
for _key, _km2 in sorted(_areas.items()):
    print(f"  {_key.replace('sigma_', '')} sigma: {_km2:8.1f} km2")
print()
print("Note aoi.mask_area_km2 reduces over aoi.analysis_region, so these areas")
print("cover the whole masked extent, not only the AOI clip.")

In [ ]:
# COLAB: RUN THIS CELL
# Map the 1-sigma and 2-sigma hot pixels over an elevation backdrop.
_backdrop = viz.elevation_backdrop(params)
_district_outline = viz.outline_image(district_fc, "222222", width=1)

for _sigma, _colour in ((1.0, "fdae61"), (2.0, "d7191c")):
    _mask = uhi_metrics.hot_pixel_mask(
        _composite, params, UTFVI_REGION, sigma=_sigma, scale_m=Z_SCALE_M
    )
    _path = viz.save_thumbnail(
        [
            _backdrop,
            _mask.selfMask().clip(UTFVI_REGION).visualize(palette=[_colour]),
            _district_outline,
        ],
        UTFVI_REGION,
        f"figures/hot_pixels_{Z_YEAR}_{_sigma:g}sigma.png",
    )
    print(f"{_sigma:g} sigma ->", _path)
    display(Image(filename=str(_path)))

## Step 8 - zonal statistics by administrative division

Any image reduced to per-division statistics. This is the table Phase 5 feeds to
`esda`/`libpysal` for Getis-Ord Gi* and Local Moran's I, and Phase 7 uses to rank
divisions for greening priority.

Two things this cell enforces:

* **GN names are not unique within Colombo District** (CLAUDE.md). The table leads
  with `adm4_pcode`; never join two of these tables on the name.
* **Running it at both GN and DS level is the MAUP sensitivity.** The same surface
  aggregated to 557 units and to 13 units gives different hot spots. Reporting one
  without the other hides a choice that changes the answer.

Requires the uploaded admin assets. If they are missing this cell will say so and
the notebook continues.

In [ ]:
# COLAB: RUN THIS CELL
ZONAL_EPOCH = EPOCHS[-1]        # the most recent epoch
ZONAL_SCALE_M = 100

_zonal_source = uhi_metrics.epoch_composite(
    UTFVI_SOURCE, params, ZONAL_EPOCH, collection=_landsat_scenes
)

division_tables = {}
for _level, _expected in (("gn", 557), ("ds", 13)):
    try:
        _table = uhi_metrics.zonal_by_division(
            _zonal_source,
            params,
            level=_level,
            scale_m=ZONAL_SCALE_M,
            reducers=("mean", "median", "stdDev"),
        )
    except Exception as exc:  # noqa: BLE001 - the asset may not be uploaded
        print(f"COULD NOT BUILD THE {_level.upper()} TABLE: {exc}")
        print("  Upload the admin assets named under 'aoi.assets' in")
        print("  config/params.yaml (notebook 01 has the step-by-step upload")
        print("  instructions), then re-run this cell.")
        continue

    division_tables[_level] = _table
    _csv = f"data/outputs/lst_by_{_level}_{ZONAL_EPOCH}.csv"
    _table.to_csv(_csv, index=False)
    print(f"{_level.upper()}: {len(_table)} divisions (expected {_expected}) -> {_csv}")
    print(f"   mean LST range {_table['mean'].min():.1f} to {_table['mean'].max():.1f} degC")
    print(f"   pixels per division: min {_table['pixel_count'].min():.0f}, "
          f"median {_table['pixel_count'].median():.0f}")
    print()

if "gn" in division_tables:
    print("Hottest 10 GN divisions (pcode is the key - GN NAMES ARE NOT UNIQUE):")
    print(division_tables["gn"].nlargest(10, "mean").round(2).to_string(index=False))

In [ ]:
# COLAB: RUN THIS CELL
# The MAUP comparison: the same surface, two aggregation units.
if len(division_tables) == 2:
    print(f"{'level':<8}{'units':>8}{'mean':>9}{'sd of unit means':>20}{'range':>10}")
    for _level, _table in division_tables.items():
        _means = _table["mean"].dropna()
        print(f"{_level.upper():<8}{len(_table):>8}{_means.mean():>9.2f}"
              f"{_means.std():>20.2f}{_means.max() - _means.min():>10.2f}")
    print()
    print("The unit means have the SAME underlying surface but a different spread")
    print("and a different range. Coarser units average away extremes, so a DS-level")
    print("hot spot map is not a downsampled GN one. Report both, or state which")
    print("unit every result is conditional on.")
else:
    print("Need both levels for the MAUP comparison; skipped.")

## Step 9 - LST against its candidate drivers

Per year: sample pixels of the dry-season composite together with NDVI, NDBI,
MNDWI and the GHSL built-up fraction, then fit OLS and Pearson correlations in
Python.

Two honest limitations, both printed with the results:

* **The p-values are anti-conservative.** Sampled pixels from a contiguous surface
  are spatially autocorrelated; OLS assumes independence, so the standard errors
  come out too small. Treat this as a screening device, not as inference. That is
  exactly why CLAUDE.md's attribution workflow continues
  `ols -> residual Moran's I -> spatial lag/error -> GWR/MGWR` in Phase 5.
* **Built-up fraction is snapped to a 5-year GHSL epoch.** A 2023 built fraction
  is really the 2020 layer.

`DRIVER_YEARS` is one round trip per year. Shorten it for a quick look.

In [ ]:
# COLAB: RUN THIS CELL
DRIVER_YEARS = list(range(params["time"]["start_year"], params["time"]["end_year"] + 1))
DRIVER_REGION = UTFVI_REGION

print(f"{len(DRIVER_YEARS)} years x "
      f"{params['uhi']['drivers']['sample_pixels']} pixels at "
      f"{params['uhi']['drivers']['sample_scale_m']} m")
print()

_t0 = time.time()
driver_coefficients, driver_corr, driver_samples = uhi_metrics.driver_series(
    UTFVI_SOURCE,
    params,
    DRIVER_REGION,
    years=DRIVER_YEARS,
    progress=True,
)
print(f"\nDone in {time.time() - _t0:.0f} s; "
      f"{len(driver_samples)} sampled pixels, "
      f"{driver_coefficients['year'].nunique() if len(driver_coefficients) else 0} years fitted.")

In [ ]:
# COLAB: RUN THIS CELL
os.makedirs("data/outputs", exist_ok=True)
driver_coefficients.to_csv("data/outputs/lst_driver_ols_by_year.csv", index=False)
driver_corr.to_csv("data/outputs/lst_driver_correlations_by_year.csv", index=False)
print("Wrote data/outputs/lst_driver_ols_by_year.csv")
print("Wrote data/outputs/lst_driver_correlations_by_year.csv")
print()

if len(driver_coefficients):
    # Mean coefficient per driver across the fitted years, with how often the
    # sign was stable - a driver that flips sign year to year is not a driver.
    _terms = driver_coefficients[driver_coefficients["term"] != "const"]
    _stability = (
        _terms.groupby("term")
        .agg(
            years=("year", "count"),
            coef_mean=("coefficient", "mean"),
            coef_sd=("coefficient", "std"),
            share_negative=("coefficient", lambda s: float((s < 0).mean())),
            median_p=("p_value", "median"),
        )
        .round(3)
    )
    print("OLS coefficients across years (degC per unit of the predictor):")
    print(_stability.to_string())
    print()
    print("R2 by year:")
    _r2 = driver_coefficients.groupby("year")["r_squared"].first().round(3)
    print(_r2.to_string())
    print()
    print("These p-values OVERSTATE significance: sampled pixels are spatially")
    print("autocorrelated, so the OLS standard errors are too small. Phase 5")
    print("tests the residual Moran's I and escalates to spatial lag/error and")
    print("then GWR/MGWR. Do not quote these p-values as final inference.")

In [ ]:
# COLAB: RUN THIS CELL
# LST against NDVI and NDBI, on the very rows the models were fitted on.
_fig = viz.plot_lst_vs_index(
    driver_samples,
    "figures/lst_vs_ndvi_ndbi.png",
    params,
    index_columns=["NDVI", "NDBI"],
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

_fig = viz.plot_lst_vs_index(
    driver_samples,
    "figures/lst_vs_mndwi_built.png",
    params,
    index_columns=["MNDWI", "built_fraction"],
)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

## What to check before signing Phase 3 off

1. **Mask areas (Step 1)** match Phase 1's verified values within a few percent:
   buffer_ring urban 37.7 / rural 206.1 km2; LCZ urban 458.5 / rural 152.2 km2.
   A larger move means the static water mask changed the masks and the SUHII
   numbers rest on different geometry than Phase 1 signed off.
2. **The SUHII table has all six sources** and both rural definitions, and the
   Aqua 2000-2002 rows are empty for the right reason (Aqua launched 2002-07),
   not because the QC filter rejected everything.
3. **Terra-night buffer_ring SUHII is positive and roughly +1 to +2 degC**
   (Step 3), consistent with the ~+2 degC CMC-vs-District nocturnal difference
   Phase 2 already measured.
4. **`terra_day` and `terra_day_relaxed` are both present** and you have compared
   them. If they disagree materially, the strict daytime series is the spatially
   biased one - say so rather than quoting it alone.
5. **The `ddof` measurement (Step 6) matched `uhi.zscore.ddof`.** If it did not,
   edit `config/params.yaml`, commit, and re-run. Record the measured value in
   `PROGRESS.md` either way so nobody repeats it.
6. **Both GN and DS zonal tables built** (Step 8), and you have looked at the MAUP
   comparison rather than only the GN table.
7. **The driver coefficients have stable signs** across years: NDVI negative,
   NDBI and built_fraction positive. A sign that flips year to year is a warning
   about the sample, not a finding.
8. **No figure or table here is labelled "air temperature"**, and nothing claims
   the UTFVI epoch maps show warming.

### Known limitations carried into Phase 4

* **UTFVI's per-year reference hides uniform warming.** The epoch maps show
  redistribution only. Phase 4's Mann-Kendall and Sen's slope are what measure
  trend, and they must not be described as confirming the UTFVI maps.
* **MODIS SUHII is a coarse-unit statistic.** At 1 km the CMC holds about 40
  pixels, and the resampled urban mask counts any 1 km pixel with *any* urban
  fraction as urban, so the urban mean is edge-contaminated.
* **Night and day are not equal-confidence.** 3 K against 1 K accepted uncertainty.
* **Driver p-values are anti-conservative** until Phase 5 corrects for spatial
  autocorrelation.
* **Built-up fraction is a 5-year epoch value**, not an annual measurement.

> Nothing in this notebook has been executed by Claude Code - it has no Earth
> Engine credentials. Until you run it, every Earth Engine cell here is unverified.